# Lab 6 Tasks - Solution

In this notebook we will analyse a dataset from an Irish triathlon by using the Pandas library. In the dataset, each row represents an athlete, described by a number of different descriptive features:

- *Number:* The athlete's race bib number
- *Place:* The place in which the athlete finished the race
- *Age:* The athlete's age
- *Gender:* The gender that the athlete declared ('M' or 'F')
- *Province:* The Irish province where the athlete comes from (Leinster, Munster, Connacht, Ulster)
- *Swim:* The time taken for the swimming segment of the event (in seconds)
- *T1:* The time taken for the first transition of the event, from cycling to swimming (in seconds)
- *Cycle:* The time taken for the cycling segment of the event (in seconds)
- *T2:* The time taken for the swimming segment of the event, from swimming to running (in seconds)
- *Run:* The time taken for the running segment of the event (in seconds)

In [3]:
import pandas as pd

## Task 1 - Data Loading and Preparation

Use Python to download a file containing triathlon dataset in CSV format from the URL:

http://mlg.ucd.ie/modules/COMP30760/triathlon.csv

Load the dataset into a Pandas DataFrame, where the row index will be given by the athlete's bib number. Display the first 20 rows of the DataFrame.

In [7]:
# we could either download the file using urllib, or we can use Pandas to do this for us
df = pd.read_csv("http://mlg.ucd.ie/modules/COMP30760/triathlon.csv", index_col="Number")

In [9]:
# display first 20 rows
df.head(20)

,Place,Gender,Age,Province,Swim,T1,Cycle,T2,Run
Number,,,,,,,,,
1,11.0,M,19,Munster,518,64.0,2293.0,42.0,1446.0
2,129.0,M,25,Leinster,1102,124.0,2662.0,47.0,1723.0
3,35.0,M,25,Leinster,693,138.0,2472.0,42.0,1466.0
4,153.0,M,26,Munster,906,318.0,3002.0,62.0,1648.0
5,34.0,M,21,Connacht,566,131.0,2518.0,65.0,1522.0
6,131.0,M,25,Connacht,940,202.0,3083.0,63.0,1381.0
7,169.0,M,23,Leinster,1118,366.0,3114.0,127.0,1674.0
8,95.0,M,22,Leinster,1002,102.0,2449.0,59.0,1756.0
9,97.0,M,29,Ulster,735,213.0,2701.0,66.0,1655.0


The dataset might contain missing values. For instance, some athletes may have registered for the race for never actually. Other athletes might have started the race, but not completed all segments of the triathlon.

From the DataFrame, identify the number of missing values in each column. Then remove any rows which contain missing values (i.e. athletes who did not fullly complete the race). How many rows are remaining?

In [12]:
# count number of missing values per column
df.isna().sum()

Place       4
Gender      0
Age         0
Province    0
Swim        0
T1          1
Cycle       1
T2          2
Run         6
dtype: int64

In [14]:
# number of rows currently
print("Data contains %d rows" % len(df))
# remove the rows with missing values
df = df.dropna()
# how many rows left?
print("Data now contains %d rows" % len(df))

Data contains 188 rows
Data now contains 182 rows


Add a new column to the DataFrame, called *Finish*, which is the total time taken for the race for each athlete (i.e. Swim + T1 + Cycle + T2 + Run).

In [17]:
# add the times for the different segments together
df["Finish"] = df["Swim"] + df["T1"] + df["Cycle"] + df["T2"] + df["Run"]

To verify the step above, sort the DataFrame, based on the *Finish* time, fastest to slowest. Display the top 10 fastest athletes overall:

In [20]:
df.sort_values(by="Finish").head(10)

,Place,Gender,Age,Province,Swim,T1,Cycle,T2,Run,Finish
Number,,,,,,,,,,
183,1.0,M,36,Ulster,577,73.0,1867.0,47.0,1195.0,3759.0
48,2.0,M,32,Munster,616,87.0,2016.0,58.0,1225.0,4002.0
54,3.0,M,35,Ulster,621,85.0,1982.0,53.0,1282.0,4023.0
92,4.0,M,49,Leinster,621,89.0,2005.0,53.0,1474.0,4242.0
118,5.0,M,52,Munster,621,104.0,2169.0,51.0,1322.0,4267.0
63,6.0,M,34,Munster,688,107.0,2197.0,60.0,1224.0,4276.0
110,7.0,M,41,Leinster,601,89.0,2188.0,49.0,1374.0,4301.0
88,8.0,M,48,Munster,877,81.0,2114.0,59.0,1192.0,4323.0
89,9.0,M,45,Munster,700,103.0,2230.0,45.0,1248.0,4326.0


## Task 2 - Initial Data Analysis

What is the average finishing time for athletes? What is the slowest finishing time?

In [24]:
# average time
print("Mean finish time = %.2f" % df["Finish"].mean())
# slowest time
print("Slowest finish time = %.2f" % df["Finish"].max())

Mean finish time = 5359.84
Slowest finish time = 7888.00


On average which segment of the race took the longest: swimming, cycling or running?

In [27]:
# calculate the averages for each segment
mean_swim = df["Swim"].mean()
mean_cycle = df["Cycle"].mean()
mean_run = df["Run"].mean()
# check which took longest
if mean_swim > mean_cycle and mean_swim > mean_run:
    print("Swimming took longest")
elif mean_cycle > mean_swim and mean_cycle > mean_run:
    print("Cycling took longest")
else:
    print("Running took longest")

Cycling took longest


How many female and male athletes competed in the race? How many athletes from each Irish province competed in the race? 

In [29]:
# create a frequency table for gender
df["Gender"].value_counts()

Gender
M    134
F     48
Name: count, dtype: int64

In [31]:
# create a frequency table for province
df["Province"].value_counts()

Province
Leinster    75
Munster     55
Ulster      30
Connacht    22
Name: count, dtype: int64

How many female and male athletes were from each of the 4 provinces? 

(Hint: We can use a cross-tabulation)

In [33]:
# apply a cross-tabulation on these two columns
pd.crosstab(df["Province"], df["Gender"])

Gender,F,M
Province,,
Connacht,6,16
Leinster,20,55
Munster,16,39
Ulster,6,24


## Task 3 - Further Data Analysis

Create a new column 'AgeCategory' that divides the runners into age categories: 16-19, 20-29, 30-39, 40-49, 50-65.

In [35]:
bins = [16, 20, 30, 40, 50, 65]
# here we specify right=False to exclude the rightmost edge
df["AgeCategory"] = pd.cut(df["Age"], bins=bins, right=False)
df.head()

,Place,Gender,Age,Province,Swim,T1,Cycle,T2,Run,Finish,AgeCategory
Number,,,,,,,,,,,
1,11.0,M,19,Munster,518,64.0,2293.0,42.0,1446.0,4363.0,"[16, 20)"
2,129.0,M,25,Leinster,1102,124.0,2662.0,47.0,1723.0,5658.0,"[20, 30)"
3,35.0,M,25,Leinster,693,138.0,2472.0,42.0,1466.0,4811.0,"[20, 30)"
4,153.0,M,26,Munster,906,318.0,3002.0,62.0,1648.0,5936.0,"[20, 30)"
5,34.0,M,21,Connacht,566,131.0,2518.0,65.0,1522.0,4802.0,"[20, 30)"


How many female and male athletes were from each age category? 

(Hint: We can use a cross-tabulation and the 'AgeCategory' column from above)

In [38]:
# apply a cross-tabulation on the next two columns
pd.crosstab(df["AgeCategory"], df["Gender"])

Gender,F,M
AgeCategory,,
"[16, 20)",0,1
"[20, 30)",12,20
"[30, 40)",19,56
"[40, 50)",13,41
"[50, 65)",4,16


What was finishing place for each athlete per province?

(Hint: We can perform group-by aggregation)

In [41]:
# first aggregate by province
groups = df.groupby("Province", observed=False)
# just select the column we want
groups["Place"].mean()

Province
Connacht    103.772727
Leinster     87.600000
Munster      93.218182
Ulster       91.266667
Name: Place, dtype: float64

What were the average times for the three segments, per age category?

(Hint: We can perform group-by aggregation)

In [43]:
# first aggregate by age category
groups = df.groupby("AgeCategory", observed=False)
# now calculate the means for the three segments
groups[["Swim", "Cycle", "Run"]].mean()

,Swim,Cycle,Run
AgeCategory,,,
"[16, 20)",518.00,2293.000000,1446.000000
"[20, 30)",839.75,2802.031250,1624.625000
"[30, 40)",836.76,2566.346667,1552.840000
"[40, 50)",888.50,2576.611111,1660.277778
"[50, 65)",870.90,2693.350000,1778.500000
